Step 1: Define File Path and Load Data
This step defines the file path for the 2022 property assessment dataset and checks if the file exists. If found, it loads the dataset, removes any leading/trailing whitespace from column names, and handles missing values to ensure consistency.

In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [10]:
# Define the file path
file_path = r'C:\Users\sul19\Desktop\701 Project\Property Assessment Datasets\Property Assessment 2022 V1.xlsx'

# Check if the file exists before attempting to load it
if os.path.exists(file_path):
    print("File found! Proceeding to load the data.")
    
    # Attempt to load the dataset using 'openpyxl' engine
    df = pd.read_excel(file_path, engine='openpyxl', na_values=['', ' '])

    # Strip any leading/trailing whitespace from the column names (just in case)
    df.columns = df.columns.str.strip()
    
else:
    print(f"File not found at {file_path}. Please check the file path.")

File found! Proceeding to load the data.


Step 2: Standardize Values in OVERALL_COND Column
This step removes non-breaking spaces and other invisible characters from the OVERALL_COND column, then replaces any blank or missing values with "none" to ensure consistency. It also standardizes specific values for improved data quality in the next steps.

In [11]:
# Replace non-breaking spaces and other invisible characters in the overall condition column
df['OVERALL_COND'] = df['OVERALL_COND'].astype(str).str.replace('\u00A0', '').str.strip()

# Replace any blank or missing values with 'none'
df['OVERALL_COND'] = df['OVERALL_COND'].replace(r'^\s*$', 'none', regex=True)

# Replace specific values in OVERALL_COND
df['OVERALL_COND'] = df['OVERALL_COND'].replace({
    'AVG - Default - Average': 'A - Average',
    'EX - Excellent': 'E - Excellent'
}, regex=False)

Step 3: Group Data by ZIP_CODE and Summarize OVERALL_COND Counts
In this step, the data is grouped by ZIP code, counting occurrences of each condition. This summary provides an overview of housing conditions across different areas based on condition counts.

In [12]:
# Group data by 'ZIP_CODE' and count the occurrences of each condition in the column
overall_cond_summary = df.groupby('ZIP_CODE')['OVERALL_COND'].value_counts().unstack().fillna(0)

# Display the result of the analysis
print("Housing condition summary by ZIP code:")
#print(condition_summary)
print(overall_cond_summary)

Housing condition summary by ZIP code:
OVERALL_COND  A - Average  E - Excellent  F - Fair  G - Good  P - Poor  \
ZIP_CODE                                                                 
2026                  5.0            0.0       0.0       1.0       0.0   
2108               1552.0          147.0       6.0     377.0       0.0   
2109               1444.0           16.0       3.0     308.0       0.0   
2110               1983.0          304.0       4.0      78.0       1.0   
2111               1888.0          314.0      23.0     484.0       0.0   
2113               1691.0            6.0      28.0     477.0       1.0   
2114               4365.0           52.0      10.0     703.0       1.0   
2115               4181.0          223.0       3.0     734.0       0.0   
2116               6866.0          357.0      26.0    1666.0       9.0   
2118               5830.0          102.0      26.0    2364.0       6.0   
2119               4392.0            4.0      86.0    1155.0      19.0   

Step 4: Map Condition Labels to Numeric Values for Analysis
To facilitate quantitative analysis, this step maps condition labels to numeric values. We then calculate the mean condition score for each ZIP code, offering insights into the average housing condition in each area.

Step 5: Convert Mean Condition Scores to Descriptive Labels
This step converts mean condition scores back to descriptive labels for better readability. Each ZIP code receives a condition label that represents the general state of housing based on its average condition score.

In [13]:
# Function to assign numerical values to conditions 
def condition_to_numeric(cond):
    mapping = {
        'E - Excellent': 5,
        'VG - Very Good': 4,
        'G - Good': 3.5,
        'A - Average': 3,
        'F - Fair': 2,
        'P - Poor': 1.5,
        'VP - Very Poor': 1,
        'US - Unsound': 0,
        
        
        'none': np.nan  # Treat 'none' as NaN for numerical purposes
    }
    return mapping.get(cond, np.nan)

# Apply the mapping to calculate average conditions
df['OVERALL_COND_NUM'] = df['OVERALL_COND'].apply(condition_to_numeric)

# Group by ZIP_CODE and calculate the mean, count, and standard deviation
condition_analysis = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Display the summary
print("Housing condition analysis by ZIP code:")
print(condition_analysis)

def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    
    else:
        return 'Unsound'

# Apply the function to map the means back to descriptive condition labels
condition_analysis['overall_cond_label'] = condition_analysis['overall_cond_mean'].apply(mean_to_condition)

# Print the result for each ZIP code
print("Overall Condition Analysis by ZIP Code:")
print(condition_analysis[['ZIP_CODE', 'overall_cond_mean', 'overall_cond_label']])

Housing condition analysis by ZIP code:
    ZIP_CODE  overall_cond_mean
0       2026           3.083333
1       2108           3.249767
2       2109           3.126032
3       2110           3.282184
4       2111           3.328905
5       2113           3.108011
6       2114           3.099116
7       2115           3.168075
8       2116           3.201668
9       2118           3.181712
10      2119           3.086398
11      2120           3.106358
12      2121           3.081120
13      2122           3.093898
14      2124           3.090200
15      2125           3.104811
16      2126           3.045493
17      2127           3.130672
18      2128           3.089984
19      2129           3.184890
20      2130           3.141362
21      2131           3.086492
22      2132           3.087807
23      2133           4.000000
24      2134           3.063827
25      2135           3.065130
26      2136           3.054513
27      2137           3.500000
28      2199           4.180556


Step 6: Standardize YR_BUILT and YR_REMODEL Columns
This step ensures YR_BUILT and YR_REMODEL columns are numeric and removes any values beyond 2024 to avoid future-dated entries. The mean construction and remodel years by ZIP code are then calculated for further analysis.

Step 7: Classify Buildings by Age
In this step, buildings are classified based on their construction or remodel year as 'Old,' 'Average,' or 'New.' This classification adds insights into the age distribution within each ZIP code, enhancing the overall housing analysis.

In [14]:
# Ensure YR_BUILT and YR_REMODEL are numeric and replace years > 2024 with NaN
df['YR_BUILT'] = pd.to_numeric(df['YR_BUILT'], errors='coerce')
df['YR_REMODEL'] = pd.to_numeric(df['YR_REMODEL'], errors='coerce')
df['YR_BUILT'] = df['YR_BUILT'].apply(lambda x: x if x <= 2024 else np.nan)
df['YR_REMODEL'] = df['YR_REMODEL'].apply(lambda x: x if x <= 2024 else np.nan)

# Group by ZIP_CODE and calculate the mean for YR_BUILT and YR_REMODEL
condition_analysis = df.groupby('ZIP_CODE').agg(
    yr_built_mean=('YR_BUILT', 'mean'),
    yr_remodel_mean=('YR_REMODEL', 'mean')
).reset_index()

# Define thresholds for building classification
def classify_building(yr_built, yr_remodel, old_threshold=1970, new_threshold=2000):
    """
    Classify building as 'Old', 'Average', or 'New' based on YR_REMODEL or YR_BUILT.
    If YR_REMODEL exists, use it; otherwise, use YR_BUILT.
    """
    if not pd.isna(yr_remodel):
        year = yr_remodel  # Use YR_REMODEL if available
    else:
        year = yr_built  # Otherwise, use YR_BUILT
    
    if pd.isna(year):
        return 'Unknown'
    elif year <= old_threshold:
        return 'Old'
    elif year >= new_threshold:
        return 'New'
    else:
        return 'Average'
    
# Apply classification based on YR_BUILT and YR_REMODEL
condition_analysis['building_classification'] = condition_analysis.apply(
    lambda row: classify_building(row['yr_built_mean'], row['yr_remodel_mean']), axis=1)

# Print the classification results based on the YR_BUILT and YR_REMODEL means
print("Building Classification Based on YR_BUILT and YR_REMODEL:")
print(condition_analysis[['ZIP_CODE', 'yr_built_mean', 'yr_remodel_mean', 'building_classification']])

Building Classification Based on YR_BUILT and YR_REMODEL:
    ZIP_CODE  yr_built_mean  yr_remodel_mean building_classification
0       2026    1956.250000      2011.000000                     New
1       2108    1907.655472      1998.741425                 Average
2       2109    1924.944077      1996.636063                 Average
3       2110    1976.613911      2001.431579                     New
4       2111    1964.384615      1999.211034                 Average
5       2113    1914.188688      1996.655154                 Average
6       2114    1930.780785      1997.237859                 Average
7       2115    1923.015891      1995.521207                 Average
8       2116    1920.314154      1999.096644                 Average
9       2118    1930.694328      2001.583211                     New
10      2119    1928.358760      2002.274657                     New
11      2120    1936.094401      2001.870927                     New
12      2121    1921.069550      2002.431597 

Step 8: Include Address Details and Finalize Output
This final step merges street address details, adds overall condition labels, assigns a constant YEAR value, and saves the output to an Excel file for further analysis.

In [15]:
# Assuming 'ST_NUM' and 'ST_NAME' are part of the original dataset
# Extract those columns from the original dataset
st_num_name = df[['ZIP_CODE', 'ST_NUM', 'ST_NAME']].drop_duplicates()

# Merge 'st_num_name' with 'condition_analysis' to include 'ST_NUM' and 'ST_NAME' with the results
condition_analysis = pd.merge(condition_analysis, st_num_name, on='ZIP_CODE', how='left')

# Add overall condition label based on previous analysis
# Ensure that the column 'overall_cond_label' from earlier condition analysis is merged correctly
overall_cond_summary = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Reapply the condition label mapping function to map numeric values to condition labels
def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    else:
        return 'Unsound'

overall_cond_summary['overall_cond_label'] = overall_cond_summary['overall_cond_mean'].apply(mean_to_condition)

# Merge overall condition labels with the condition_analysis DataFrame
condition_analysis = pd.merge(condition_analysis, overall_cond_summary[['ZIP_CODE', 'overall_cond_label']], on='ZIP_CODE', how='left')

# Assign a constant value for 'YEAR'
condition_analysis['YEAR'] = 2022

# Rearranging the columns as requested
final_output = condition_analysis[['YEAR', 'ST_NUM', 'ST_NAME', 'ZIP_CODE', 'overall_cond_label', 'building_classification']]

# Save the final result to a new Excel file
output_file_path = 'Property_Assessment_2022_Output.xlsx'
final_output.to_excel(output_file_path, index=False)

print(f"File saved successfully to {output_file_path}")

File saved successfully to Property_Assessment_2022_Output.xlsx
